# Administrer les formulaires par l'API — le formulaire est un contenu, pas une table

Troisieme notebook de la serie « AI Engine par son API ». Apres le socle
(grain 1) et les chatbots (grain 2), une autre fonctionnalite coeur :
les **formulaires** (AI Forms). Ou le grain 2 montrait un chatbot
configurable comme document JSON global, ce notebook revele une
architecture differente dans le meme plugin : un formulaire AI Engine
est un **custom post type** — un contenu WordPress (`mwai_form`) —
manipule par un CRUD unitaire, dont le corps est du **contenu
Gutenberg** et dont le rendu public passe par un **shortcode**.

Le cycle complete est demontre par l'API : lister, creer (une coquille
vide), remplir et publier, rendre public dans une page, supprimer. Et
une limite honnete, mesuree : ce que la version gratuite rend, et ce
qui reste derriere la version Pro.

> Les sorties de ce notebook proviennent d'une execution reelle contre
> l'instance locale (voir « Provenance et limites », en fin de fichier).


## La serie « AI Engine par son API »

Le projet Livres Agites a mis AI Engine au coeur d'une maison d'edition :
bot d'accueil, agents d'ateliers, bibliothecaire documentee par RAG,
formulaires dynamiques. Cette serie presente le plugin de maniere
reproductible — **sans jamais exposer de donnees client** :

| Notebook | Contenu |
|----------|---------|
| `presenter-ai-engine-par-son-api` | instance, API, catalogue des chatbots, premiere completion |
| `configurer-chatbots-par-l-api` | lire, dupliquer, ecrire et interroger des chatbots (document JSON global) |
| `administrer-les-formulaires-par-l-api` (ce notebook) | le formulaire comme contenu : CRUD, publication, rendu public |
| notebooks suivants | RAG/embeddings, agents MCP, ... |

Trois niveaux de lecture, dans chaque notebook :

1. **Decouverte** — ce que fait la fonctionnalite, vue par l'API ;
2. **Branchement** — comment on l'a branchee dans le projet ;
3. **Exercice** — reutiliser le pattern sur un cas voisin.


In [1]:
# Configuration et helpers. Aucune cle ni adresse de provider n'est stockee
# dans ce fichier : tout vient de instance-jetable/.env (README, etape 5).

import base64
import os
from pathlib import Path

import requests
from dotenv import load_dotenv

# Localisation du .env : a cote du notebook (instance-jetable/.env),
# sinon dans le repertoire courant.
charges = []
for candidat in (Path("instance-jetable/.env"), Path(".env")):
    if candidat.exists():
        load_dotenv(candidat)
        charges.append(str(candidat))
print("Fichiers .env charges :", charges or "(aucun)")

BASE_URL = os.getenv("VALMONT_BASE_URL", "http://localhost:8093").rstrip("/")
ADMIN_USER = os.getenv("VALMONT_ADMIN_USER", "")
APP_PASSWORD = os.getenv("VALMONT_APP_PASSWORD", "")
print("Base URL :", BASE_URL)


def api(route, method="GET", payload=None, params=None):
    """Appel REST WordPress. route est relative, ex. '/mwai/v1/forms/list'."""
    url = BASE_URL + "/wp-json" + route
    entetes = {"Content-Type": "application/json"}
    if ADMIN_USER and APP_PASSWORD:
        creds = base64.b64encode(f"{ADMIN_USER}:{APP_PASSWORD}".encode()).decode()
        entetes["Authorization"] = "Basic " + creds
    reponse = requests.request(method, url, headers=entetes, json=payload,
                               params=params, timeout=120)
    reponse.raise_for_status()
    return reponse.json()


def http_get(chemin):
    """GET public (sans authentification) — pour tester le rendu visiteur."""
    reponse = requests.get(BASE_URL + chemin, timeout=60)
    reponse.raise_for_status()
    return reponse


Fichiers .env charges : ['instance-jetable\\.env']
Base URL : http://localhost:8093


## Deux fonctionnalites, deux styles d'API

Le catalogue des routes (grain 1) cachait une asymetrie instructive.
Comparer la famille des chatbots et celle des formulaires :

| | Chatbots (grain 2) | Formulaires (ce notebook) |
|---|---|---|
| Nature du stockage | entree d'une **liste** dans les options du plugin | **custom post type** WordPress (`mwai_form`) |
| Ecriture | `POST /settings/chatbots` : la liste **entiere** est remplacee | `POST /forms/update` : **un** formulaire, par `id` |
| Creation | ajouter un element a la liste envoyee | `POST /forms/create` : alloue une coquille vide |
| Suppression | retirer l'element de la liste et re-POSTer | `POST /forms/delete` : une route dediee par `id` |
| Modele mental | document de configuration | **contenu** (titre, corps, statut, publication) |

Le meme plugin expose deux styles d'API selon la nature de l'objet.
Un chatbot se comporte comme un reglage ; un formulaire se comporte
comme un article. Ce n'est pas un detail : cela dicte les operations
possibles (versionner, publier, brouillonner un formulaire comme un
article) et les pieges (remplacer la liste des chatbots ecrase tout ;
oublier de publier un formulaire le laisse invisible).


In [2]:
# 1. Lister : l'etat des formulaires (vide sur une instance neuve)
liste = api("/mwai/v1/forms/list")["forms"]
print("Formulaires existants :", [(f["id"], f["title"], f["status"]) for f in liste])


Formulaires existants : [(5, 'Soumission de manuscrit', 'publish')]


Sur l'instance neuve du grain 1, cette liste etait vide. C'est la
premiere lecon de ce notebook : « pas de formulaire » n'est pas un
etat fige, c'est un etat d'entree — et l'API permet de le remplir.

## Creer : une coquille, pas un formulaire

`POST /forms/create` n'accepte qu'un `title` — et rend une coquille :
un identifiant alloue, un titre, un **contenu vide**, un statut
**draft**. Envoyer des champs dans le payload de creation est ignore :
la creation est une **allocation**, pas une definition. Le contenu
vient ensuite, par la route d'update — exactement comme on cree un
article vide avant de l'ecrire.


In [3]:
# 2. Creer : la coquille (title honore, contenu vide, statut draft)
coquille = api("/mwai/v1/forms/create", method="POST",
               payload={"title": "Coquille de demonstration"})["form"]
print("id alloue      :", coquille["id"])
print("titre          :", coquille["title"]["raw"])
print("champs envoyes :", "ignores par la route (seul title est lu)")

detail = api("/mwai/v1/forms/get", params={"id": coquille["id"]})["form"]
print("contenu brut   :", repr(detail["content"]["raw"]))
print("statut         :", detail["status"])


id alloue      : 9
titre          : Coquille de demonstration
champs envoyes : ignores par la route (seul title est lu)
contenu brut   : ''
statut         : draft


## Remplir et publier : le formulaire devient un contenu

`POST /forms/update` ecrit `title`, `content` et `status`. Le contenu
d'un formulaire est du **contenu WordPress** — ici des blocs
Gutenberg. On construit la fiche de soumission de la Maison Valmont,
on la publie (`publish`), et on verifie la persistance par une
relecture.

Pour que le notebook soit rejouable, la creation suit le pattern
**upsert par titre** : si « Soumission de manuscrit » existe deja, on
le reutilise ; sinon, on alloue une coquille. Puis on ecrit toujours
le contenu — meme idempotent.


In [4]:
# 3. Upsert : trouver ou allouer le formulaire principal, puis ecrire
TITRE = "Soumission de manuscrit"

forms = api("/mwai/v1/forms/list")["forms"]
existant = next((f for f in forms if f["title"] == TITRE), None)
if existant:
    FORM_ID = existant["id"]
    print("Reutilise (upsert) :", TITRE, "id =", FORM_ID)
else:
    FORM_ID = api("/mwai/v1/forms/create", method="POST",
                  payload={"title": TITRE})["form"]["id"]
    print("Alloue :", TITRE, "id =", FORM_ID)

CONTENU = (
    "<!-- wp:paragraph -->\n<p>Soumettez votre manuscrit a la Maison Valmont. "
    "Indiquez le titre, le genre et collez les cent premieres lignes.</p>\n<!-- /wp:paragraph -->\n"
    "<!-- wp:paragraph -->\n<p>Reponse du comite sous six semaines.</p>\n<!-- /wp:paragraph -->"
)

api("/mwai/v1/forms/update", method="POST", payload={
    "id": FORM_ID, "title": TITRE, "content": CONTENU, "status": "publish",
})
print("Ecrit et publie :", FORM_ID)

# Persistance : relecture apres ecriture
relu = api("/mwai/v1/forms/get", params={"id": FORM_ID})["form"]
print("statut persiste :", relu["status"])
print("contenu persiste :", repr(relu["content"]["raw"][:60]), "...")


Reutilise (upsert) : Soumission de manuscrit id = 5


Ecrit et publie : 5


statut persiste : publish
contenu persiste : '<!-- wp:paragraph -->\n<p>Soumettez votre manuscrit a la Mais' ...


## Le rendu public : un shortcode, comme pour un article

Un formulaire publie ne vit pas seul dans une page : il s'insere par
le **shortcode** `[mwai_form id=N]`. On cree une page par l'API REST
standard de WordPress (`/wp/v2/pages` — pas une route AI Engine), on y
place le shortcode, puis on interroge la page **en visiteur
non authentifie** et on cherche le texte du formulaire dans le HTML
rendu. La boucle est bouclee : cree par l'API, ecrit par l'API, rendu
au public.


In [5]:
# 4. Rendre public : page avec shortcode, puis verification en visiteur
SLUG = "soumettre-un-manuscrit"
page = api("/wp/v2/pages", params={"slug": SLUG})
if page:
    PAGE_ID = page[0]["id"]
    print("Page existante reutilisee : " + str(PAGE_ID))
else:
    PAGE_ID = api("/wp/v2/pages", method="POST", payload={
        "title": "Soumettre un manuscrit", "status": "publish",
        "content": "<!-- wp:shortcode -->\n[mwai_form id=\"" + str(FORM_ID) + "\"]\n<!-- /wp:shortcode -->",
    })["id"]
    print("Page creee : " + str(PAGE_ID))

html = http_get("/" + SLUG + "/").text
present = "Soumettez votre manuscrit" in html
print("Page publique HTTP 200, formulaire rendu :", present)

# Mesure du rendu : le contenu est-il du texte affiche ou des champs ?
fragment = html[html.find("Soumettez votre manuscrit"):]
print("Balises <input> dans le rendu :", fragment.count("<input"))
print("Blocs <p> dans le rendu       :", fragment.count("<p"))


Page existante reutilisee : 6
Page publique HTTP 200, formulaire rendu : True
Balises <input> dans le rendu : 0
Blocs <p> dans le rendu       : 3


## Ce que dit la mesure : gratuit rend du contenu, Pro rend des champs

La mesure ci-dessus est le coeur de l'observation honnete de ce
notebook : sur la version gratuite (3.7.0, wordpress.org), le
formulaire rendu est du **contenu** (des paragraphes) — zero champ de
saisie. Les formulaires *pilotes par l'IA* — champs dynamiques,
validation par modele, conditions — vivent dans la version Pro : le
code gratuit n'enregistre que deux shortcodes (`mwai_form` et
`mwai_chatbot`, verifie dans le plugin au moment du developpement),
sans les shortcodes de champs individuels.

Ce n'est pas une limite cachee : c'est une frontiere de produit,
mesurable par l'API et par le rendu. La lecon pour le projet : ce que
le comparatif appelle « AI Forms (text/image/audio/file avec logique
conditionnelle) » suppose la version Pro — sur la gratuite, l'API
administrative existe deja (c'est ce notebook), mais le formulaire
n'est pas encore un formulaire.

## Supprimer : une route dediee, un retour a l'etat initial

`POST /forms/delete` supprime par `id` — pas de remplacement de liste
comme pour les chatbots. On supprime la coquille de demonstration de
la cellule 2 et on verifie la liste finale : il ne reste que le
formulaire principal, publie.


In [6]:
# 5. Supprimer la coquille, verifier l'etat final
api("/mwai/v1/forms/delete", method="POST", payload={"id": coquille["id"]})
print("Coquille supprimee :", coquille["id"])

final = api("/mwai/v1/forms/list")["forms"]
print("Etat final :", [(f["id"], f["title"], f["status"]) for f in final])


Coquille supprimee : 9


Etat final : [(5, 'Soumission de manuscrit', 'publish')]


## Ce qu'on en a fait dans le projet Livres Agites

Dans le projet d'origine (AI Engine Pro), les formulaires sont la
porte d'entree du workflow editorial : soumission de manuscrit avec
champs conditionnels, pieces jointes, validations. Le parcours 4
(`livresagites-parcours.md`) les decrit comme une **machine a etats**
— un formulaire a logique de branchement a plus de chemins que de
champs, et trois grandeurs emergent (chemins atteignables, cout LLM,
champs morts). Le compagnon stdlib
[`auditer-un-formulaire-conditionnel.ipynb`](auditer-un-formulaire-conditionnel.ipynb)
enumere ces chemins sur un fixture synthetique. Ce notebook en est la
face administrative : comment un formulaire vit dans WordPress —
coquille, contenu, publication, rendu, suppression — avant meme que
la logique conditionnelle n'entre en jeu.


## Exercices

Trois exercices, du plus simple au plus integre. Les fonctions sont a
completer ; `api()` et les donnees des cellules precedentes sont
disponibles. Chaque exercice se verifie d'une ligne de test.


### Exercice 1 — lire un formulaire proprement

Completez `lire_formulaire(form_id)` : elle retourne le document du
formulaire (id, titre, statut), ou `None` s'il n'existe pas — sans
lever d'exception.


In [7]:
def lire_formulaire(form_id):
    """Retourne le document du formulaire form_id, None s'il est absent."""
    # A COMPLETER : forms/get avec params={'id': ...} et gestion du cas absent
    return None


### Exercice 2 — creer un formulaire complet en deux temps

Completez `creer_formulaire(titre, contenu)` : elle alloue la
coquille (create), l'ecrit et la publie (update), puis retourne
l'identifiant — le pattern create-shell-then-update de ce notebook.


In [8]:
def creer_formulaire(titre, contenu):
    """Alloue, ecrit et publie ; retourne l'id du formulaire cree."""
    # A COMPLETER : forms/create puis forms/update (status='publish')
    return None


### Exercice 3 — purger les brouillons

Completez `purger_brouillons()` : elle liste les formulaires,
supprime ceux au statut `draft` et retourne la liste des identifiants
supprimes. Attention a ne toucher QUE les drafts.


In [9]:
def purger_brouillons():
    """Supprime tous les formulaires au statut draft ; retourne leurs ids."""
    # A COMPLETER : forms/list, filtrer status=='draft', forms/delete sur chacun
    return []


## Provenance et limites

- **Instance testee** : `http://localhost:8093`, montee via
  `instance-jetable/docker-compose.jetable.example.yml`, AI Engine 3.7.0
  (version gratuite, wordpress.org), corpus synthetique « Maison Valmont ».
- **Determinisme** : contrairement aux notebooks 1 et 2, aucune
  completion LLM ici — toutes les sorties sont deterministes au rejeu
  pres des identifiants alloues (l'upsert par titre evite les doublons).
- **Frontiere gratuite/Pro** : mesuree par le rendu (paragraphes sans
  champs) et par lecture du code du plugin au developpement (deux
  shortcodes enregistres : `mwai_form`, `mwai_chatbot`). Les champs IA
  dynamiques sont une fonctionnalite Pro.
- **Endpoints verifies ici (firsthand)** : `/mwai/v1/forms/list`,
  `/forms/create`, `/forms/get`, `/forms/update`, `/forms/delete`,
  `/wp/v2/pages` (creation + lecture publique).
- **Frontieres** : pas de donnees client, pas de secret, pas d'IP de
  provider — regles du chantier CoursIA.
